In [1]:
print("hello world")

hello world


In [2]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

# Continue coding here
# Use as many cells as you like

In [3]:
# ----------------------------------------
# 2. Load datasets
# ----------------------------------------

nasdaq100_ca = pd.read_csv("nasdaq100_CA_practice.csv")

price_change = pd.read_csv(
    "nasdaq100_price_change_practice.csv"
)


In [ ]:
# ----------------------------------------
# 3. Add YTD performance
# ----------------------------------------

nasdaq100_ca = nasdaq100_ca.merge(
    price_change[["symbol", "ytd"]],
    on="symbol",
    how="inner"
)


print("Combined DataFrame:")
print(nasdaq100_ca.head())



In [5]:
# ----------------------------------------
# 4. Classify companies into sectors
# ----------------------------------------

sectors = [
    "Technology",
    "Consumer Cyclical",
    "Industrials",
    "Utilities",
    "Healthcare",
    "Communication",
    "Energy",
    "Consumer Defensive",
    "Real Estate",
    "Financial"
]

nasdaq100_ca["sector"] = ""


for index, row in nasdaq100_ca.iterrows():

    prompt = f"""
Classify the following company into exactly ONE of these sectors:

Technology
Consumer Cyclical
Industrials
Utilities
Healthcare
Communication
Energy
Consumer Defensive
Real Estate
Financial

Company name: {row["name"]}
Stock symbol: {row["symbol"]}

Return ONLY the sector name.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    sector = response.choices[0].message.content.strip()

    nasdaq100_ca.loc[index, "sector"] = sector



In [6]:
# ----------------------------------------
# 5. Display sector counts
# ----------------------------------------

print("\nCompanies by sector:")

print(
    nasdaq100_ca["sector"].value_counts()
)


Companies by sector:
sector
Technology            31
Healthcare             6
Consumer Cyclical      5
Industrials            4
Communication          3
Energy                 2
Consumer Defensive     1
Utilities              1
Name: count, dtype: int64


In [7]:
# ----------------------------------------
# 6. Ask OpenAI for recommendations
# ----------------------------------------

company_data = nasdaq100_ca[
    ["symbol", "name", "headQuarter", "ytd", "sector"]
].to_string(index=False)


prompt = f"""
Analyze the following NASDAQ-100 company data.

The data contains:
- Company name
- Stock symbol
- Headquarters
- YTD performance
- Sector

Identify the TWO best-performing sectors based primarily
on YTD performance.

Then recommend at least TWO companies from each of
the two best sectors.

For each recommended company, explain briefly why
it is a good candidate based on the supplied data.

Do not invent performance numbers.

Company data:

{company_data}
"""


response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)


In [8]:
# ----------------------------------------
# 7. Store recommendations
# ----------------------------------------

stock_recommendations = response.choices[0].message.content

print("\nStock Recommendations:")
print(stock_recommendations)


Stock Recommendations:
To identify the two best-performing sectors based on Year-To-Date (YTD) performance, we will analyze the provided data. 

### Step 1: Calculate Average YTD Performance by Sector

1. **Technology Sector**:
   - AAPL: -12.00
   - ADBE: -2.34
   - ADI: 2.49
   - AMAT: 7.32
   - AMD: 12.15
   - ANSS: 21.81
   - APP: 26.64
   - ARM: 31.47
   - ASML: 36.30
   - AVGO: 41.13
   - AXON: 45.96
   - CDNS: 55.62
   - CRWD: 89.43
   - CSCO: -10.74
   - CSGP: -5.91
   - CTSH: 3.75
   - DDOG: 13.41
   - EA: 23.07
   - FTNT: 42.39
   - GOOGL: 56.88
   - INTC: 76.20
   - INTU: 81.03
   - KLAC: 90.69
   - LRCX: -4.65
   - MCHP: 5.01
   - MDB: 9.84
   - META: 14.67
   - MRVL: 19.50
   - MSFT: 24.33
   - MU: 29.16

   **Average YTD Performance**: 
   - Total YTD:  0.00 (sum of all YTD values)
   - Count: 27
   - Average: 0.00 / 27 = **0.00**

2. **Healthcare Sector**:
   - AMGN: 16.98
   - DXCM: 18.24
   - GILD: 47.22
   - IDXX: 66.54
   - ILMN: 71.37
   - ISRG: 85.86

   **Average